# Section 3: Robustness, Where You Go Beyond (28 pts)

ECON 5200 Midterm Project · Replication Study

**AI is allowed in this notebook**, as a documented co-pilot under P.R.I.M.E. File every exchange, in full, in [`ai-appendix.md`](../ai-appendix.md) under the exercise it belongs to. If you cannot explain a line, you cannot submit it.

At least four exercises the original did not run. Choose from modern estimators, a specification curve, sensitivity to unobserved confounding, inference (clustering at the level of treatment assignment, a wild cluster bootstrap if clusters are few), bandwidth or instrument strength, and placebo tests, or justify an alternative. For each one: what you did, what you found, and what it implies about the original claim.

In [ ]:
# Clone and Install Cell:
# Set directory to cloned github repository for working inside Google Colab.
# Runs Replication Notebook 1 in the background to import
# output/headline_table.csv
# Imports rdrobust for exercise 4
# Only uncomment this cell if running the notebook from Google Colab under a new runtime

#!git clone https://github.com/Zach-Kaplan44/ECON-5200-Replication-Study /content/ECON-5200-Replication-Study
# %cd /content/ECON-5200-Replication-Study/notebooks

# from pathlib import Path
# if not Path("/content/ECON-5200-Replication-Study/output/headline_table.csv").exists():
#    !jupyter nbconvert --to notebook --execute /content/ECON-5200-Replication-Study/notebooks/01_positioning_and_replication.ipynb --output /tmp/01_run.ipynb

# !pip install rdrobust


Cloning into '/content/ECON-5200-Replication-Study'...
remote: Enumerating objects: 42, done.
remote: Counting objects: 100% (42/42), done.
remote: Compressing objects: 100% (39/39), done.
remote: Total 42 (delta 14), reused 10 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (42/42), 2.96 MiB | 2.99 MiB/s, done.
Resolving deltas: 100% (14/14), done.
/content/ECON-5200-Replication-Study/notebooks
[NbConvertApp] Converting notebook /content/ECON-5200-Replication-Study/notebooks/01_positioning_and_replication.ipynb to notebook
0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=

## Setup

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as sm

# The repository root, whether this runs from notebooks/ (Jupyter) or from the root (reproduce.py).
ROOT = Path.cwd() if (Path.cwd() / "download_data.py").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
(ROOT / "output").mkdir(exist_ok=True)

from download_data import MY_PAPER, PAPERS, fetch

## Data

In [ ]:
# Set MY_PAPER once, in download_data.py. fetch() downloads into data/ only if a file is missing.
if MY_PAPER:
    paths = fetch(MY_PAPER)
    print(PAPERS[MY_PAPER]["paper"])
    dwi = pd.read_stata(paths[0])
else:
    print("TODO: set MY_PAPER in download_data.py (castle, card, lee, dw, hansen or abortion).")

  data/hansen_dwi.dta  9.5 MB
Hansen (2015), punishment and deterrence in drunk driving


## The four exercises

In [ ]:
# Function Creating And Data Prep Cell
# Rebuilds the variables from the replication section
# Creates a function to re-run my replication regressions with options
# For robustness checks, and checks that the numbers match my replication

CUTOFF = 0.08

# Converts low_score into the same units as BAC
dwi['bac_min'] = dwi['low_score'] / 1000

# Age and year fixed effects, as in Hansen's controls
age_dummy = pd.get_dummies(dwi['aged'], drop_first=True, prefix='Age').astype(int)
year_dummy = pd.get_dummies(dwi['year'], drop_first=True, prefix='Year').astype(int)
dwi[age_dummy.columns] = age_dummy
dwi[year_dummy.columns] = year_dummy

RUNNING = 'bac_min'
H = 0.05
KERNEL = "rectangular"
CONTROLS = ['male', 'white'] + list(age_dummy.columns) + list(year_dummy.columns)

def rd_estimate(
    data,                   # the full dataset; never modified
    cutoff,                 # cutoff in integer thousandths of BAC, e.g. 80 or 150
    h,                      # bandwidth in the same integer units, e.g. 50 or 25
    controls=None,          # list of control column names, or None for no controls
    se="cluster",           # "cluster" (grouped by bac_min) or "HC1"
    rule=None,              # "ge", "gt", or None to use the default rule below
    exclude_years=None,     # optional iterable of years to drop before windowing
    outcome="recidivism",
    score="low_score",
):
    """Local linear RD at `cutoff` with a rectangular kernel of half-width `h`.

    Returns (fitted statsmodels result, jump coefficient as an unrounded float).
    """
    # Default treatment rule: the statute defines aggravated DUI as a BAC
    # strictly above 0.15 (Hansen, Table 1), so strictly above at that cutoff
    # and at-or-above everywhere else.
    if rule is None:
        rule = "gt" if cutoff == 150 else "ge"
    if rule not in ("ge", "gt"):
        raise ValueError("rule must be 'ge', 'gt', or None")

    # Year exclusions apply before the window so the bandwidth isn't affected
    # by which rows the year filter removes.
    sample = data
    if exclude_years is not None:
        # isin() against numbers silently matches nothing if year is stored as
        # strings or categories, which would drop no rows and give a wrong
        # answer with no warning, so check the dtype first.
        if not pd.api.types.is_numeric_dtype(data["year"]):
            raise TypeError(
                f"exclude_years requires a numeric 'year' column, but it has "
                f"dtype {data['year'].dtype}. Convert it first, e.g. "
                f"data['year'] = pd.to_numeric(data['year'])."
            )
        sample = sample[~sample["year"].isin(list(exclude_years))]

    # Window on the integer score, both edges included. Comparing integers is
    # exact; a float comparison on BAC can drop boundary rows, as it does at the
    # 0.15 window, where 0.200 - 0.150 exceeds 0.05 in float64.
    window = sample[np.abs(sample[score] - cutoff) <= h].copy()

    # Running variable in BAC units, then recentered so the constant is the
    # intercept at the cutoff and above_cutoff is the jump.
    window["bac_min"] = window[score] / 1000
    window["bac_centered"] = window["bac_min"] - cutoff / 1000

    # Treatment indicator, also built on the integer score to avoid float error
    # at 0.150, where the >= comparison on bac_min is unreliable.
    if rule == "gt":
        window["above_cutoff"] = (window[score] > cutoff).astype(int)
    else:
        window["above_cutoff"] = (window[score] >= cutoff).astype(int)

    # Interaction term: lets the slope differ on either side of the cutoff.
    window["above_centered"] = window["above_cutoff"] * window["bac_centered"]

    # Rectangular kernel = plain unweighted OLS on the rows inside the window.
    X = window[["above_cutoff", "bac_centered", "above_centered"]]

    if controls:
        # Drop controls that are constant in this sample (e.g. year dummies for
        # years that were excluded, or age dummies absent from a narrow window);
        # they're collinear with the constant and would make X singular.
        keep = [c for c in controls if window[c].nunique(dropna=False) > 1]
        X = pd.concat([X, window[keep]], axis=1)

    X = sm.add_constant(X)
    y = window[outcome]

    # Hansen clusters on the running variable; HC1 offered as a robustness check.
    if se == "cluster":
        fit = sm.OLS(y, X).fit(cov_type="cluster",
                               cov_kwds={"groups": window["bac_min"]})
    elif se == "HC1":
        fit = sm.OLS(y, X).fit(cov_type="HC1")
    else:
        raise ValueError("se must be 'cluster' or 'HC1'")

    return fit, float(fit.params["above_cutoff"])

In [ ]:
import statsmodels.api as sm

# Check Cell
# Ensure the function matches the same numbers as my replication
# Expected values come from the saved headline table, not from in-memory
# variables, so this cell runs in a fresh notebook.
table_path = ROOT / "output" / "headline_table.csv"
if not table_path.exists():
    raise FileNotFoundError(
        f"{table_path} not found. Run notebook 01 or `python reproduce.py` "
        f"to generate the headline table before running these checks."
    )

# Row labels were saved as the index, so read them back as the index.
headline = pd.read_csv(table_path, index_col=0)

# Each entry: row label in the table, and the kwargs that should reproduce it.
checks = [
    ("Table 3, Panel A, col 1: BAC >= 0.08, h = 0.05",
     dict(cutoff=80, h=50, controls=CONTROLS)),
    ("Table 3, Panel B, col 1: BAC >= 0.08, h = 0.025",
     dict(cutoff=80, h=25, controls=CONTROLS)),
    ("Appendix Table 1, Panel A, col 1: BAC >= 0.08, h = 0.05, no controls",
     dict(cutoff=80, h=50, controls=None)),
    ("Appendix Table 1, Panel B, col 1: BAC >= 0.08, h = 0.025, no controls",
     dict(cutoff=80, h=25, controls=None)),
    ("Table 4, Panel A, col 1: BAC > 0.15, h = 0.05",
     dict(cutoff=150, h=50, controls=CONTROLS)),
    ("Table 4, Panel B, col 1: BAC > 0.15, h = 0.025",
     dict(cutoff=150, h=25, controls=CONTROLS)),
    ("Appendix Table 2, Panel A, col 1: BAC > 0.15, h = 0.05, no controls",
     dict(cutoff=150, h=50, controls=None)),
    ("Appendix Table 2, Panel B, col 1: BAC > 0.15, h = 0.025, no controls",
     dict(cutoff=150, h=25, controls=None)),
]

# Snapshot the columns to confirm the function didn't touch the input.
cols_before = list(dwi.columns)

print(f"{'setting':<40}{'function':>12}{'saved':>12}{'diff':>12}{'N':>9}  match")
all_match = True
for label, kwargs in checks:
    expected = float(headline.loc[label, "my_estimate"])  # saved unrounded
    fit, jump = rd_estimate(dwi, **kwargs)                # defaults: cluster SEs, default rule
    diff = jump - expected
    match = np.isclose(jump, expected, rtol=0, atol=1e-10)
    all_match &= match
    short = label.split(": ")[1]                          # trim the table reference for width
    print(f"{short:<40}{jump:>12.6f}{expected:>12.6f}{diff:>12.2e}"
          f"{int(fit.nobs):>9}  {'yes' if match else 'NO'}")

cols_after = list(dwi.columns)
print(f"\nall eight reproduce: {all_match}")
print(f"dwi columns unchanged: {cols_before == cols_after}")

setting                                     function       saved        diff        N  match
BAC >= 0.08, h = 0.05                      -0.022145   -0.022145   -7.98e-17    93899  yes
BAC >= 0.08, h = 0.025                     -0.019260   -0.019260   -6.59e-17    50016  yes
BAC >= 0.08, h = 0.05, no controls         -0.021845   -0.021845   -3.12e-17    93899  yes
BAC >= 0.08, h = 0.025, no controls        -0.019402   -0.019402   -4.51e-17    50016  yes
BAC > 0.15, h = 0.05                       -0.009019   -0.009019   -5.72e-17   139033  yes
BAC > 0.15, h = 0.025                      -0.009922   -0.009922   -2.08e-17    75909  yes
BAC > 0.15, h = 0.05, no controls          -0.009279   -0.009279   -3.82e-17   139033  yes
BAC > 0.15, h = 0.025, no controls         -0.009693   -0.009693   -4.16e-17    75909  yes

all eight reproduce: True
dwi columns unchanged: True


### Exercise 1: Heteroskeadsticity-robust (HC1) Standard Errors

**What I did.** Hansen clusters SE at 0.001 BAC value to allow for correlation among drivers with the same reading. I estimated using heteroskedasticity-robust SEs (HC1) instead.

In [ ]:
fit_hc1, jump_hc1 = rd_estimate(dwi, cutoff=80, h=50, controls=CONTROLS, se="HC1")
print(f"{jump_hc1:.4f} (HC1 SE {fit_hc1.bse['above_cutoff']:.4f})")

# Fresh names throughout so nothing from the earlier cells is overwritten.
inference_rows = []          # one dict per specification, assembled into the table below
coef_mismatches = []         # labels where the two coefficients aren't bit-identical

for spec_label, spec_kwargs in checks:
    # Same specification, fit twice; `se` is passed separately so the dicts
    # inside `checks` are never mutated.
    fit_cl, jump_cl = rd_estimate(dwi, se="cluster", **spec_kwargs)
    fit_rb, jump_rb = rd_estimate(dwi, se="HC1", **spec_kwargs)

    # Only the covariance estimator differs, so the point estimates come from
    # the same OLS solve and should be exactly equal, not merely close.
    if jump_cl != jump_rb:
        coef_mismatches.append(spec_label)

    # Pull the SE and p-value for the jump coefficient out of each fit.
    se_cluster = float(fit_cl.bse["above_cutoff"])
    se_hc1 = float(fit_rb.bse["above_cutoff"])

    # conf_int() returns a frame indexed by parameter with columns 0 (lower)
    # and 1 (upper); default alpha=0.05 gives the 95% interval. Each fit builds
    # its interval from its own covariance estimator, so the two differ in width
    # but are centered on the same coefficient.
    ci_cl = fit_cl.conf_int().loc["above_cutoff"]
    ci_rb = fit_rb.conf_int().loc["above_cutoff"]

    inference_rows.append({
        "label": spec_label,
        "coef": jump_cl,                       # identical under both, checked above
        "se_cluster": se_cluster,
        "se_hc1": se_hc1,
        "se_ratio": se_hc1 / se_cluster,       # >1 means clustering tightens the SE
        "p_cluster": float(fit_cl.pvalues["above_cutoff"]),
        "p_hc1": float(fit_rb.pvalues["above_cutoff"]),
        "ci_low_cluster": float(ci_cl[0]),
        "ci_high_cluster": float(ci_cl[1]),
        "ci_low_hc1": float(ci_rb[0]),
        "ci_high_hc1": float(ci_rb[1]),
        "N": int(fit_cl.nobs),                 # window is the same, so N matches too
    })

# Row labels as the index, matching how the headline table is saved.
inference_df = pd.DataFrame(inference_rows).set_index("label")

# Report the coefficient check before anything else, since a mismatch would
# mean the two runs weren't actually the same specification.
if coef_mismatches:
    raise AssertionError(
        "Coefficient changed with the SE type in: " + "; ".join(coef_mismatches)
    )
print("coefficients identical under cluster and HC1: True\n")

# Write the table out alongside the other saved output.
inference_path = ROOT / "output" / "robustness_inference.csv"
inference_path.parent.mkdir(parents=True, exist_ok=True)   # no-op if output/ exists
inference_df.to_csv(inference_path)                        # full precision on disk
print(f"saved to {inference_path}\n")

# Print rounded to four decimals; N stays an integer because float_format
# only applies to float columns.
print(inference_df.to_string(float_format=lambda v: f"{v:.4f}"))

-0.0221 (HC1 SE 0.0043)
coefficients identical under cluster and HC1: True

saved to /content/ECON-5200-Replication-Study/output/robustness_inference.csv

                                                                         coef  se_cluster  se_hc1  se_ratio  p_cluster  p_hc1  ci_low_cluster  ci_high_cluster  ci_low_hc1  ci_high_hc1       N
label                                                                                                                                                                                          
Table 3, Panel A, col 1: BAC >= 0.08, h = 0.05                        -0.0221      0.0041  0.0043    1.0542     0.0000 0.0000         -0.0302          -0.0141     -0.0306      -0.0137   93899
Table 3, Panel B, col 1: BAC >= 0.08, h = 0.025                       -0.0193      0.0048  0.0057    1.1844     0.0001 0.0007         -0.0287          -0.0099     -0.0304      -0.0081   50016
Appendix Table 1, Panel A, col 1: BAC >= 0.08, h = 0.05, no controls  -0.0218

**What I found.** At the 0.08 cutoff, HC1 SEs are 4-18% larger than Hansen's clustered SEs, and the gap is largest at the narrower bandwidth. For the headline number, the 95% interval widens somewhat, from (-3.02, -1.41) pp with clustered SE, to (-3.06, -1.37) with HC1 SE. At 0.15, the two methods are very close, with ratios of just 0.98 and 1.02.
**What it implies about the original claim.** Hansen's precise estimation of effects does not depend on the choice to cluster by BAC.  Clustered SE in RD when a running variable is discrete can't reliably account for specification error which can be fixed using heteroskedasticity-robust SE. Under the robust SEs, each estimate still keeps its significance level. Their critique is a valid one generally, but does not change anything about the original claim. The precision of the effect at 0.15 with the narrow bandwidth is still modest as in the original paper, with significance at the 5% level, but not the 1% level.

### Exercise 2: Placebo test on Recidivism
**What I did.** The current thresholds are based on Washingsto DUI laws. I estimated the same specifications at six BAC values where the Washington law has no impact, 0.040, 0.050, 0.110, 0.115, 0.120, and 0.180, to see if it is possible that the jump in recidivism happens even without the DUI laws. I used a bandwidth of 0.025 throughout, which prevents the real thresholds from falling inside of the placebo windows.

In [ ]:
fit_p50, jump_p50 = rd_estimate(dwi, cutoff=50, h=25, controls=CONTROLS)
print(f"{jump_p50:.4f} (SE: {fit_p50.bse['above_cutoff']:.4f})")

# All new names so nothing from the earlier cells (including fit_p50) is touched.
PLACEBO_H = 25                       # same bandwidth for every row, in thousandths
REAL_CUTOFFS = (80, 150)             # the two statutory thresholds
placebo_cutoffs = (40, 50, 110, 115, 120, 180)
placebo_rows = []                    # one dict per cutoff, assembled into the table below

# Run the real cutoffs alongside the placebos so the table is self-contained;
# sorted so the rows read left to right along the BAC scale.
placebo_specs = sorted(
    [(c, "placebo") for c in placebo_cutoffs] + [(c, "real") for c in REAL_CUTOFFS]
)

for cut, kind in placebo_specs:
    # A placebo window that straddles a real threshold would pick up the actual
    # treatment effect leaking in, so the row would be meaningless. Check the
    # window bounds before fitting, and only for the placebos.
    if kind == "placebo":
        contaminating = [r for r in REAL_CUTOFFS if abs(r - cut) <= PLACEBO_H]
        if contaminating:
            raise ValueError(
                f"Placebo cutoff {cut} with h={PLACEBO_H} spans the window "
                f"[{cut - PLACEBO_H}, {cut + PLACEBO_H}], which contains the real "
                f"cutoff(s) {contaminating}. Narrow the bandwidth or drop this cutoff."
            )

    # Identical specification throughout: h=25, CONTROLS, clustered SEs. The
    # treatment rule falls out of the function's default, so the placebos use
    # >= and the 0.15 row uses >, as in the headline table.
    fit_pl, jump_pl = rd_estimate(dwi, cutoff=cut, h=PLACEBO_H, controls=CONTROLS)

    # 95% interval for the jump, built from this fit's clustered covariance.
    ci_pl = fit_pl.conf_int().loc["above_cutoff"]

    placebo_rows.append({
        "cutoff_bac": cut / 1000,                          # report in BAC units
        "kind": kind,                                      # "real" or "placebo"
        "coef": jump_pl,
        "se": float(fit_pl.bse["above_cutoff"]),
        "p_value": float(fit_pl.pvalues["above_cutoff"]),
        "ci_low": float(ci_pl[0]),
        "ci_high": float(ci_pl[1]),
        "N": int(fit_pl.nobs),
    })

# Index on the cutoff so rows are identified by the thing that varies.
placebo_df = pd.DataFrame(placebo_rows).set_index("cutoff_bac")

# Save unrounded, matching how the other output tables are written.
placebo_path = ROOT / "output" / "robustness_placebo.csv"
placebo_path.parent.mkdir(parents=True, exist_ok=True)     # no-op if output/ exists
placebo_df.to_csv(placebo_path)
print(f"saved to {placebo_path}\n")

# Display only: four decimals for the estimate columns, three for p-values.
# Per-column formatters, since to_string's float_format is applied globally.
placebo_formatters = {
    "coef":    lambda v: f"{v:.4f}",
    "se":      lambda v: f"{v:.4f}",
    "ci_low":  lambda v: f"{v:.4f}",
    "ci_high": lambda v: f"{v:.4f}",
    "p_value": lambda v: f"{v:.3f}",
}
print(placebo_df.to_string(formatters=placebo_formatters))

-0.0184 (SE: 0.0137)
saved to /content/ECON-5200-Replication-Study/output/robustness_placebo.csv

               kind    coef     se p_value  ci_low ci_high      N
cutoff_bac                                                       
0.040       placebo  0.0214 0.0142   0.131 -0.0063  0.0491   9727
0.050       placebo -0.0184 0.0137   0.178 -0.0452  0.0084  16417
0.080          real -0.0193 0.0048   0.000 -0.0287 -0.0099  50016
0.110       placebo  0.0013 0.0043   0.763 -0.0072  0.0098  76985
0.115       placebo -0.0011 0.0036   0.759 -0.0083  0.0060  79001
0.120       placebo  0.0017 0.0035   0.622 -0.0052  0.0086  80194
0.150          real -0.0099 0.0046   0.030 -0.0189 -0.0010  75909
0.180       placebo -0.0026 0.0050   0.595 -0.0124  0.0071  55597


**What I found.** At the real cutoffs, the estimates with bandwidth .025 are -1.93 pp in receidivism at 0.08 (p < 0.001), and -0.99 pp at 0.15 (p - 0.030). For the placebo thresholds of 0.110, 0.115, 0.120, and 0.180, the placebo estimates range from -0.26 to +0.17 pp, with p-values ranging between 0.60-0.76, with the windows raning between ~56,000-80,000 stops. With SEs ranging between 0.35-0.50 similar to those at the reaul cutoffs (0.46-0.48), so the test could detect a jump of the real effect's size if it were there. Below 0.08, the placebos at 0.040 and 0.050 are not significant (p = 0.13, 0.18) but have large estimates of +2.14 and -1.84 pp. Since few drivers (9,700 and 16,400 stops) are stopped at those windows, there is an SE of about 1.4 pp.

**What it implies about the original claim.** Where the test has a sufficiently large enough sample within the window, the method finds that there is only notable jumps at 0.08 and 0.15. This supports that the drops in recidivism at 0.08 and 0.15 are effects of the punishments rather than coming from fitting straight lines to a curved relationship, something that Hansen does not report. Below 0.08, the data is too thin for the test to rule out jumps as large as the real effect.

### Exercise 3: Checking strong or soft comparison

**What I did.** I checked if the estimate around the 0.15 threshold depends on a coding choice Hansen leaves somewhat ambiguous. Hansen defines an aggravated DUI as strictly above 0.150. He describes both thresholds using the word "above" however at the 0.08 threshold it is used to mean "at or above" and at the 0.15 threshold it is used to mean "strictly above". My original estimate used a strict >0.150 at the upper threshold and this compares that with 0.150 being counted as treated using a >= comparison.

In [ ]:
fit_ge, jump_ge = rd_estimate(dwi, cutoff=150, h=50, controls=CONTROLS, rule='ge')

# All new names so nothing from the earlier cells (including fit_ge) is touched.
rule015_path_in = ROOT / "output" / "headline_table.csv"
if not rule015_path_in.exists():
    raise FileNotFoundError(
        f"{rule015_path_in} not found. Run notebook 01 or `python reproduce.py` "
        f"to generate the headline table before running this check."
    )

# Row labels were saved as the index, same as in the reproduction cell.
rule015_headline = pd.read_csv(rule015_path_in, index_col=0)

# The four 0.15 specifications: each label paired with the kwargs that produce it.
# The labels read "BAC > 0.15" because the headline table was built with rule="gt";
# the "ge" rows below deliberately depart from that coding.
rule015_specs = [
    ("Table 4, Panel A, col 1: BAC > 0.15, h = 0.05",
     dict(cutoff=150, h=50, controls=CONTROLS)),
    ("Table 4, Panel B, col 1: BAC > 0.15, h = 0.025",
     dict(cutoff=150, h=25, controls=CONTROLS)),
    ("Appendix Table 2, Panel A, col 1: BAC > 0.15, h = 0.05, no controls",
     dict(cutoff=150, h=50, controls=None)),
    ("Appendix Table 2, Panel B, col 1: BAC > 0.15, h = 0.025, no controls",
     dict(cutoff=150, h=25, controls=None)),
]

rule015_rows = []        # one dict per (specification, rule) pair
rule015_repro_fail = []  # labels where the "gt" run misses the saved my_estimate
rule015_n_fail = []      # labels where N differs between the two rules

for spec_label, spec_kwargs in rule015_specs:
    # Hansen's own number and my saved replication number for this specification.
    paper_est = float(rule015_headline.loc[spec_label, "paper_estimate"])
    mine_est = float(rule015_headline.loc[spec_label, "my_estimate"])

    n_by_rule = {}  # collect N per rule so the two can be compared after the inner loop

    for rule_name in ("gt", "ge"):
        # Same window and same controls both times; only the treatment coding
        # at low_score == 150 changes. `rule` is passed separately so the dicts
        # in rule015_specs stay reusable.
        fit_r, jump_r = rd_estimate(dwi, rule=rule_name, **spec_kwargs)

        # The "gt" run is the headline coding, so it must land on my saved
        # estimate exactly; a miss means the function or the data has drifted.
        if rule_name == "gt" and not np.isclose(jump_r, mine_est, rtol=0, atol=1e-10):
            rule015_repro_fail.append(spec_label)

        n_by_rule[rule_name] = int(fit_r.nobs)

        # 95% interval for the jump, from this fit's clustered covariance.
        ci_r = fit_r.conf_int().loc["above_cutoff"]

        rule015_rows.append({
            "label": spec_label,
            "rule": rule_name,
            "coef": jump_r,
            "se": float(fit_r.bse["above_cutoff"]),
            "p_value": float(fit_r.pvalues["above_cutoff"]),
            "ci_low": float(ci_r[0]),
            "ci_high": float(ci_r[1]),
            "N": int(fit_r.nobs),
            "paper_estimate": paper_est,
            "gap_from_paper": jump_r - paper_est,   # signed: positive means my estimate is larger
        })

    # The window is defined by low_score alone, so recoding treatment can't
    # change which rows are in the sample. A difference here would mean the
    # rule is leaking into the window filter.
    if n_by_rule["gt"] != n_by_rule["ge"]:
        rule015_n_fail.append(spec_label)

# Index on the specification and the rule, so each row is uniquely identified.
rule015_df = pd.DataFrame(rule015_rows).set_index(["label", "rule"])

# Raise on either failure before writing, so a bad run leaves no stale CSV.
if rule015_repro_fail:
    raise AssertionError(
        'rule="gt" did not reproduce my_estimate in: ' + "; ".join(rule015_repro_fail)
    )
if rule015_n_fail:
    raise AssertionError(
        "N differed between rules in: " + "; ".join(rule015_n_fail)
    )
print('rule="gt" reproduces my_estimate in all four specifications: True')
print("N identical under both rules in all four specifications: True")

# The stops whose treatment status actually flips between the two codings.
# If this is zero, the two rules are the same regression and every pair of
# rows below will be identical.
rule015_n_at_150 = int((dwi["low_score"] == 150).sum())
print(f"stops with low_score == 150 (treatment flips under 'ge'): {rule015_n_at_150}\n")

# Save unrounded, matching the other output tables.
rule015_path_out = ROOT / "output" / "robustness_rule015.csv"
rule015_path_out.parent.mkdir(parents=True, exist_ok=True)   # no-op if output/ exists
rule015_df.to_csv(rule015_path_out)
print(f"saved to {rule015_path_out}\n")

# Display only: four decimals for the estimate columns, three for p-values.
# Per-column formatters, since to_string's float_format applies to every float.
rule015_formatters = {
    "coef":           lambda v: f"{v:.4f}",
    "se":             lambda v: f"{v:.4f}",
    "ci_low":         lambda v: f"{v:.4f}",
    "ci_high":        lambda v: f"{v:.4f}",
    "gap_from_paper": lambda v: f"{v:.4f}",
    "paper_estimate": lambda v: f"{v:.4f}",
    "p_value":        lambda v: f"{v:.3f}",
}
print(rule015_df.to_string(formatters=rule015_formatters))

rule="gt" reproduces my_estimate in all four specifications: True
N identical under both rules in all four specifications: True
stops with low_score == 150 (treatment flips under 'ge'): 1527

saved to /content/ECON-5200-Replication-Study/output/robustness_rule015.csv

                                                                             coef     se p_value  ci_low ci_high       N paper_estimate gap_from_paper
label                                                                rule                                                                             
Table 4, Panel A, col 1: BAC > 0.15, h = 0.05                        gt   -0.0090 0.0034   0.008 -0.0157 -0.0023  139033        -0.0100         0.0010
                                                                     ge   -0.0073 0.0035   0.035 -0.0141 -0.0005  139033        -0.0100         0.0027
Table 4, Panel B, col 1: BAC > 0.15, h = 0.025                       gt   -0.0099 0.0046   0.030 -0.0189 -0.0010   75909       

**What I found.** 1,527 stops in the sample have a BAC of exactly 0.150 which is 1.1% of the window using h=0.05 and 2% using h=0.025. Counting these as aggravated shrinks every 0.15 estimate, by 19% at h=0.05 (-0.90 to -0.73 pp with controls) and about 30% at h=0.025(-0.99 to -0.70 pp). At the narrower bandwidth, the effect loses significance (p = 0.130 with controls and 0.148 without), with intervals that include 0.

**What it implies about the original claim.** Hansen's public estimates are consistent with treating it as a hard >0.15 which the statute does, and it also implies he did not use >= coding. The shrink in the effect also implies the design is correct in predicting thta the law drives the effect. Drivers at 0.150 exactly receive an ordinary DUI, so counting them as treated dilutes this jump, meaning recidivism indeed changes exactly at the legal threshold, between 0.150 and 0.151. It also shows that the 0.15 effect is somewhat fragile, as it shows that reclassifying just 1-2% of its sample moves it by up to a third, and at the narrower bandwidth determines its significance.

### Exercise 4: Checking estimate using rdrobust.
**What I did.** Since the analysis in Hansen's paper was done before the rdrobust method was well known, I estimate the findings using rdrobust to check if there is bias to Hansen's estimate.

In [ ]:

from rdrobust import rdrobust
def rd(cutoff, h=None, kernel="triangular", controls=None, data=None, outcome="recidivism", donut=0.0):
    data = dwi if data is None else data
    d = data[(data[RUNNING] - cutoff).abs() >= donut]
    covs = d[controls] if controls else None
    kw = {"h": h} if h else {}
    r = rdrobust(y=d[outcome], x=d[RUNNING], c=cutoff, p=1, kernel=kernel, covs=covs, **kw)
    return {"jump": r.coef.iloc[0, 0], "se": r.se.iloc[0, 0], "p_robust": r.pv.iloc[2, 0],
            "h": r.bws.iloc[0, 0], "n": int(r.N_h[0] + r.N_h[1])}


def show(label, r):
    print(f"{label:<34} {r['jump']:+.4f} (SE {r['se']:.4f}, robust p {r['p_robust']:.3f}), h = {r['h']:.3f}, n = {r['n']:,}")

r08 = rd(0.08, controls=CONTROLS, data=dwi[dwi['low_score'] <=150])
show("0.08, MSE-optimal h, controls", r08)

# All new names so nothing from the earlier cells (r08, rd, show) is touched.
rdr_rows = []   # one dict per (cutoff, controls) combination

# Each entry: label, the cutoff rdrobust centers on, the row restriction that
# keeps this window from reaching the other threshold, and the controls.
#   - 0.08 runs on low_score <= 150, so the window can't cross the aggravated cutoff.
#   - the aggravated threshold is centered at 0.1505 rather than 0.150, which puts
#     a BAC of exactly 0.150 on the untreated side as the statute requires, and
#     runs on low_score >= 80 so the window can't reach back to 0.08.
rdr_specs = [
    ("0.08, controls",        0.08,   dwi["low_score"] <= 150, CONTROLS),
    ("0.08, no controls",     0.08,   dwi["low_score"] <= 150, None),
    ("0.1505, controls",      0.1505, dwi["low_score"] >= 80,  CONTROLS),
    ("0.1505, no controls",   0.1505, dwi["low_score"] >= 80,  None),
]

for rdr_label, rdr_cut, rdr_mask, rdr_controls in rdr_specs:
    # Apply the row restriction; .copy() so nothing downstream touches dwi.
    d = dwi[rdr_mask].copy()

    # Covariates as a frame, or None when running without controls. Constant
    # columns are dropped for the same reason as in rd_estimate: a year or age
    # dummy with no variation in this subsample is collinear with the intercept.
    if rdr_controls:
        covs = d[[c for c in rdr_controls if d[c].nunique(dropna=False) > 1]]
    else:
        covs = None

    # Same settings as rd(): local linear, triangular kernel, MSE-optimal
    # bandwidth chosen by rdrobust itself (no h passed), RUNNING as the score.
    # Called directly rather than through rd() so the full result object is
    # available, not just the five fields rd() returns.
    r = rdrobust(y=d["recidivism"], x=d[RUNNING], c=rdr_cut, p=1,
                 kernel="triangular", covs=covs)

    # rdrobust stacks three rows of inference: 0 conventional, 1 bias-corrected,
    # 2 robust. The point estimate is the conventional one; the robust row
    # carries the bias-corrected estimate with its wider, bias-aware interval,
    # and is the row rdrobust intends you to report inference from.
    rdr_rows.append({
        "label": rdr_label,
        "cutoff": rdr_cut,
        "controls": rdr_controls is not None,
        "coef": float(r.coef.iloc[0, 0]),            # conventional point estimate
        "se": float(r.se.iloc[0, 0]),                # conventional SE
        "p_conventional": float(r.pv.iloc[0, 0]),
        "ci_low": float(r.ci.iloc[0, 0]),            # conventional 95% interval
        "ci_high": float(r.ci.iloc[0, 1]),
        "coef_bc": float(r.coef.iloc[1, 0]),         # bias-corrected estimate
        "se_robust": float(r.se.iloc[2, 0]),         # robust SE
        "p_robust": float(r.pv.iloc[2, 0]),          # the p-value rd() reports
        "ci_low_robust": float(r.ci.iloc[2, 0]),     # robust 95% interval
        "ci_high_robust": float(r.ci.iloc[2, 1]),
        # bws row 0 is the main bandwidth h, row 1 the bias bandwidth b; the two
        # columns are the left and right sides, which can differ.
        "h_left": float(r.bws.iloc[0, 0]),
        "h_right": float(r.bws.iloc[0, 1]),
        "b_left": float(r.bws.iloc[1, 0]),
        "b_right": float(r.bws.iloc[1, 1]),
        # N_h counts effective observations inside h on each side.
        "n_left": int(r.N_h[0]),
        "n_right": int(r.N_h[1]),
        "N": int(r.N_h[0] + r.N_h[1]),
    })

# Index on the label so each row is identified by its specification.
rdr_df = pd.DataFrame(rdr_rows).set_index("label")

# Save unrounded, matching the other output tables.
rdr_path = ROOT / "output" / "robustness_rdrobust.csv"
rdr_path.parent.mkdir(parents=True, exist_ok=True)   # no-op if output/ exists
rdr_df.to_csv(rdr_path)
print(f"saved to {rdr_path}\n")

# Display only: four decimals for estimates and intervals, three for p-values
# and bandwidths. Per-column formatters, since to_string's float_format is global.
rdr_formatters = {
    "coef":           lambda v: f"{v:+.4f}",
    "se":             lambda v: f"{v:.4f}",
    "ci_low":         lambda v: f"{v:+.4f}",
    "ci_high":        lambda v: f"{v:+.4f}",
    "coef_bc":        lambda v: f"{v:+.4f}",
    "se_robust":      lambda v: f"{v:.4f}",
    "ci_low_robust":  lambda v: f"{v:+.4f}",
    "ci_high_robust": lambda v: f"{v:+.4f}",
    "p_conventional": lambda v: f"{v:.3f}",
    "p_robust":       lambda v: f"{v:.3f}",
    "h_left":         lambda v: f"{v:.3f}",
    "h_right":        lambda v: f"{v:.3f}",
    "b_left":         lambda v: f"{v:.3f}",
    "b_right":        lambda v: f"{v:.3f}",
    "cutoff":         lambda v: f"{v:.4f}",
}
print(rdr_df.to_string(formatters=rdr_formatters))

Mass points detected in the running variable.
0.08, MSE-optimal h, controls      -0.0168 (SE 0.0062, robust p 0.034), h = 0.025, n = 48,199
Mass points detected in the running variable.
Mass points detected in the running variable.
Mass points detected in the running variable.
Mass points detected in the running variable.
saved to /content/ECON-5200-Replication-Study/output/robustness_rdrobust.csv

                    cutoff  controls    coef     se p_conventional  ci_low ci_high coef_bc se_robust p_robust ci_low_robust ci_high_robust h_left h_right b_left b_right  n_left  n_right      N
label                                                                                                                                                                                           
0.08, controls      0.0800      True -0.0168 0.0062          0.007 -0.0290 -0.0047 -0.0158    0.0075    0.034       -0.0304        -0.0012  0.025   0.025  0.040   0.040   15371    32828  48199
0.08, no controls  

**What I found.** The MSE optimal bandwidth is about 0.025 at both cutoffs, which is half of Hansen's normal 0.05. At 0.08, at 0.08, the estimate is -1.68 pp with controls and -1.58 without, and a robust 95% interval of (-3.04, -0.12) (p=0.034). This is about 25% smaller than the replication headline estimate of -2.21, and close to Hansen's own triangular-kernel estimate at h=0.025 (-1.5) At the 0.15 threshold, the estimate is -1.24 pp with controls -1.31 with a robust 95% interval of (-2.51, -0.11) (p=0.033). This is larger than my section 2 estimate (-0.90) and slightly above Hansen's (-1.0). The bias correction is small, only about 0.1 points at each cutoff.
**What it implies about the original claim.** Both effects survive a method that was developed after the paper in both sign and significance, but have weaker precision. They are significant at the 5% level, but not the 1% level that Hansen reports. At 0.08, the size of the effect depends on how heavily the estimate weights drivers near the cutoff, from about -2.2 pp at Hansen's headline bandwidth, to about -1.6 pp with rdrobust's data-driven bandwidth and triangular kernel. Since the bias correction is fairly samll, the difference reflects how wide a window the estimate draws on rather than curvature at the cutoff, which is consistent with Hansen's appendix figure 3. The 0.08 effect is then better described as roughly 1.6-2.2 pp points with the headline number being at the upper end. This matches the paper's abstract phrasing that crossing the 0.08 threshold reduces recidivism "by up to 2 pp". The 0.15 effect, however, holds at Hansen's size. Since data-driven bandwidths and bias-corrected inference come after Hansen's analysis, and he reported the narrow bandwidth and full range in his appendix, this is a test of how his results hold up under current practice rather than evidence that he made an incorrect choice.